# 00 - Colab setup for QUEST-KG (CIKM 2026)

Run this once per Colab session before any experiment notebook.

1. Mount Google Drive (data, models, checkpoints persist here).
2. Clone the GitHub repo for the latest code.
3. Install dependencies with CUDA wheels matching Colab's PyTorch.
4. Verify CUDA device.
5. Load HuggingFace token from Colab's secret manager.
6. Paste the anti-idle JavaScript into the browser console to keep the session alive.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = '/content/drive/MyDrive/quest_kg'
for sub in ['data/raw', 'data/processed', 'models', 'ckpts', 'results', 'logs', 'figures']:
    pathlib.Path(f'{ROOT}/{sub}').mkdir(parents=True, exist_ok=True)
print(os.listdir(ROOT))

## 2. Clone (or update) the repo

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/AKSB0567/quest-kg-cikm2026.git'
REPO_DIR = '/content/quest-kg-cikm2026'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

## 4. Verify CUDA

In [ ]:
from quest_kg.utils.device import cuda_info
info = cuda_info()
assert info['cuda_available'], 'No CUDA device! Runtime > Change runtime type > GPU.'

## 5. HuggingFace login (needed for gated LLaMA models)

**Safer setup (recommended):** Add your HF token to Colab's secret manager - DO NOT paste it directly into a notebook cell or share it in chat.

1. Click the **key icon** in the left sidebar of Colab.
2. Click **+ Add new secret**.
3. Name: `HF_TOKEN`, Value: your token from https://huggingface.co/settings/tokens (Read scope).
4. Toggle **Notebook access** ON for this notebook.

Tokens in the secret manager are encrypted by Google, never written to notebook output, and not copied when you share the notebook.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token)
    print('logged in via Colab secret manager')
except Exception as e:
    print(f'secret manager unavailable ({e}); falling back to interactive prompt')
    import getpass
    login(token=getpass.getpass('Paste your HF token (will be masked): '))

## 6. Anti-idle JavaScript (paste into browser console)

Colab Pro disconnects idle sessions after a while and has no background execution.
Open the browser DevTools console (F12 in Chrome) and paste:

```javascript
function ClickConnect(){
  console.log('keepalive');
  document.querySelector('colab-toolbar-button#connect')?.click();
}
setInterval(ClickConnect, 60000);
```

This clicks the reconnect button every 60s. Combined with our checkpoint-resume
pattern, this keeps long-running experiments alive across the 12-hour session cap.